# EEG Classification - All Three Models

This notebook demonstrates all three EEG classification models:
1. **CSP** - Common Spatial Patterns with LDA/Random Forest
2. **EEGNet** - Deep CNN for EEG classification  
3. **CBraMod** - Transformer foundation model

Perfect for Google Colab testing and learning!

In [ ]:
# Mount Google Drive (for Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Navigate to project directory
%cd /content/drive/MyDrive/eye-eeg/eye-eeg

In [ ]:
# Install required packages for Colab
import sys
if 'google.colab' in sys.modules:
    %pip install -q tensorflow scikit-learn mne torch einops

In [ ]:
# Import all necessary libraries
sys.path.append("..")
import numpy as np
from data.data_configs import SUBJECTS, CBRAMOD_CHANNELS, COLAB_LABELED_EEG_DIR
from data.data_loader import DataLoader
from classification.training_split import split_normal

# Import our unified classifiers
from classification.models.csp.csp_classifier import CSPClassifier
from classification.models.eeg_net.eegnet_classifier import EEGNetClassifier  
from classification.models.cbramod.cbramod_classifier import CBraModClassifier

In [ ]:
# Load EEG data
print("Loading EEG data...")
X, y, subject_ids = DataLoader.load_labeled_eeg_data(
    subjects=SUBJECTS, 
    channels=CBRAMOD_CHANNELS, 
    labeled_eeg_data_dir=COLAB_LABELED_EEG_DIR
)
print(f"Data shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class distribution: {np.bincount(y)}")

In [ ]:
# Split data into train/validation/test sets
X_train, X_val, X_test, y_train, y_val, y_test = split_normal(X, y)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 1. CSP Classifier

Common Spatial Patterns with bandpass filtering and LDA/Random Forest classification.

In [ ]:
# Train CSP classifier
print("🧠 Training CSP Classifier...")
csp = CSPClassifier(n_components=6, clf_type='lda')
csp_results = csp.train(X_train, y_train, X_val, y_val, X_test, y_test, subject_ids=subject_ids)

## 2. EEGNet Classifier

Deep CNN with depthwise separable convolutions for automatic feature learning.

In [ ]:
# Train EEGNet classifier
print("\n🤖 Training EEGNet Classifier...")
eegnet = EEGNetClassifier(dropoutRate=0.5, F1=8)
eegnet_results = eegnet.train(X_train, y_train, X_val, y_val, X_test, y_test, 
                             subject_ids=subject_ids, epochs=50)

## 3. CBraMod Classifier

Transformer foundation model with pretrained weights for complex temporal patterns.

In [ ]:
# Train CBraMod classifier
print("\n🔬 Training CBraMod Classifier...")
cbramod = CBraModClassifier(dropout=0.1)
cbramod_results = cbramod.train(X_train, y_train, X_val, y_val, X_test, y_test, 
                               subject_ids=subject_ids, epochs=25, lr=5e-5)

## Results Comparison

Compare performance of all three models.

In [ ]:
# Compare results
print("\n" + "="*50)
print("🏆 FINAL RESULTS COMPARISON")
print("="*50)
print(f"CSP:     {csp_results['balanced_acc']:.4f} balanced accuracy")
print(f"EEGNet:  {eegnet_results['balanced_acc']:.4f} balanced accuracy") 
print(f"CBraMod: {cbramod_results['balanced_acc']:.4f} balanced accuracy")
print("\nAUROC Scores:")
print(f"CSP:     {csp_results.get('auroc', 'N/A')}")
print(f"EEGNet:  {eegnet_results.get('auroc', 'N/A')}")
print(f"CBraMod: {cbramod_results.get('auroc', 'N/A')}")

## Test Predictions

Make predictions on test data with each model.

In [ ]:
# Get predictions from all models
print("Making predictions on test data...")

# Prepare test data for each model (they handle this automatically)
csp_test_preds = csp.predict(X_test)
eegnet_test_preds = eegnet.predict(X_test) 
cbramod_test_preds = cbramod.predict(X_test)

print("Predictions completed!")
print(f"CSP predictions: {np.bincount(csp_test_preds.astype(int))}")
print(f"EEGNet predictions: {np.bincount(eegnet_test_preds.astype(int))}")
print(f"CBraMod predictions: {np.bincount(cbramod_test_preds.astype(int))}")
print(f"True labels: {np.bincount(y_test)}")

## Model Information

Summary of each model's characteristics and use cases.

In [ ]:
print("📊 MODEL SUMMARY")
print("="*50)
print("CSP Classifier:")
print("  • Method: Spatial filtering + LDA/RF")
print("  • Best for: Motor imagery, simple features")
print("  • Fast training, interpretable")
print()
print("EEGNet Classifier:")
print("  • Method: Depthwise separable CNN")
print("  • Best for: General EEG classification")
print("  • Automatic feature learning")
print()
print("CBraMod Classifier:")
print("  • Method: Transformer foundation model")
print("  • Best for: Complex temporal patterns")
print("  • Pretrained weights, transfer learning")

## Next Steps

- Adjust hyperparameters for better performance
- Try different data preprocessing
- Use parameter optimization methods
- Analyze model predictions and errors
- Experiment with different channel selections